# 01 Consumer Purchase Behavior

**Research question:** What patterns characterize consumer purchasing behavior on the Olist marketplace?
**Sources:** `analytics.fact_orders` (order grain), `staging.orders`+`staging.customers` (customer_unique_id linkage). **Filter:** all `order_status` values (approved KPI default). **KPIs:** `order_count`, `gmv_payments`, `aov`, `repeat_customer_rate`, `cancellation_rate` (see `docs/olist/03_kpi_definitions.md`).

In [1]:

from src.db import get_connection

conn = get_connection(read_only=True)


In [2]:
# Grain: order (analytics.fact_orders), all statuses
orders = conn.execute("""
SELECT COUNT(DISTINCT order_id) AS order_count,
       SUM(order_total_value) AS gmv_payments,
       AVG(order_total_value) AS aov,
       quantile_cont(order_total_value, 0.5) AS median_order_value,
       SUM(CASE WHEN order_status='canceled' THEN 1 ELSE 0 END) AS canceled_orders
FROM analytics.fact_orders
""").fetchdf()
orders['cancellation_rate'] = orders['canceled_orders'] / orders['order_count']
orders

,order_count,gmv_payments,aov,median_order_value,canceled_orders,cancellation_rate
0,99441,16008872.12,160.988648,105.29,625.0,0.006285


In [3]:
# Repeat rate (grain: customer_unique_id via staging linkage), all statuses
repeat = conn.execute("""
WITH orders_per AS (
  SELECT cu.customer_unique_id, COUNT(DISTINCT o.order_id) AS cnt
  FROM staging.orders o
  JOIN staging.customers cu ON o.customer_id = cu.customer_id
  GROUP BY cu.customer_unique_id
)
SELECT COUNT(*) AS customers,
       SUM(CASE WHEN cnt > 1 THEN 1 ELSE 0 END) AS repeat_customers,
       1.0*SUM(CASE WHEN cnt > 1 THEN 1 ELSE 0 END)/COUNT(*) AS repeat_customer_rate
FROM orders_per
""").fetchdf()
repeat

,customers,repeat_customers,repeat_customer_rate
0,96096,2997.0,0.031188


In [4]:
# Monthly volume (order grain, all statuses)
monthly = conn.execute("""
SELECT strftime(order_purchase_timestamp, '%Y-%m') AS month,
       COUNT(DISTINCT order_id) AS order_count,
       SUM(order_total_value) AS gmv_payments
FROM analytics.fact_orders
GROUP BY 1 ORDER BY 1
""").fetchdf()
monthly.tail()

,month,order_count,gmv_payments
20,2018-06,6167,1023880.50
21,2018-07,6292,1066540.75
22,2018-08,6512,1022425.32
23,2018-09,16,4439.54
24,2018-10,4,589.67
